[Lab README](README.md)

# Lab 2.3: Text2Cypher, optional

Every retriever so far returns top-k chunks. Ask any of them how many hotels
have a swimming pool and the honest answer is that they cannot know: they saw
five chunks, and the count lives in the whole graph.

`Text2CypherRetriever` writes a Cypher query instead of searching, and lets
Neo4j compute the answer over everything that matches. It is the right tool for
aggregation, and it is the one pattern in this lab with a model in the query
path, which is exactly why it is optional and why it is last.

Skip this notebook if you are short on time. Nothing later in the workshop
depends on it.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install -r requirements.txt

print("Environment ready")

## Connect and verify the graph

Retrieval notebooks do not create schema artifacts. This cell requires both Lab 1
indexes to be online with the expected label, property, dimensions, and
similarity function, then checks the graph facts the questions below depend on.

In [ ]:
import os

import boto3
from dotenv import load_dotenv
from IPython.display import HTML, display
from neo4j import GraphDatabase

load_dotenv()

NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in NEO4J_VARS if not os.environ.get(name)]
has_aws = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = not missing and has_aws

if missing:
    print(f"Neo4j is not configured: set {', '.join(missing)} in the repo-root .env")
if not has_aws:
    print("No AWS credentials found, so the Bedrock calls below cannot run")

if RETRIEVAL_READY:
    # Imported here because `workshop.graph_connection` raises at import when
    # NEO4J_PASSWORD is unset, which would fail this cell instead of skipping it.
    from workshop.graph_connection import NEO4J_URI, neo4j_auth
    from workshop.retrieval_contract import (
        CHUNK_FULLTEXT_INDEX,
        CHUNK_VECTOR_INDEX,
        EMBEDDING_DIMENSIONS,
        EMBEDDING_MODEL_ID,
    )
    from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

    driver = GraphDatabase.driver(NEO4J_URI, auth=neo4j_auth())
    driver.verify_connectivity()

    try:
        verify_retrieval_indexes(driver)
        problems = fixture_problems(driver)
        if problems:
            raise RuntimeError("; ".join(problems))
    except Exception as exc:
        raise RuntimeError(
            f"The graph is not ready for retrieval: {exc}\n"
            "Run Lab 1 (01-graph-build/1.1_build_graph.ipynb) first."
        ) from exc

    print(f"{CHUNK_VECTOR_INDEX} is ONLINE")
    print(f"{CHUNK_FULLTEXT_INDEX} is ONLINE")
    print("Every graph fact these questions depend on is present")
else:
    print("\nThe cells below will skip. Finish Lab 0 and Lab 1, then come back.")

## The schema the graph actually holds

Lab 1 pinned this schema during extraction, so the traversals below can name
relationships instead of discovering them. Render it before using any retriever
that traverses, so the shape of the added context is predictable.

In [ ]:
from workshop.graph_schema import GRAPH_SCHEMA

pattern_rows = "".join(
    f"<tr><td><strong>{source}</strong></td><td>&mdash;{relationship}&rarr;</td>"
    f"<td><strong>{target}</strong></td></tr>"
    for source, relationship, target in GRAPH_SCHEMA["patterns"]
)
display(HTML(
    "<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>"
    f"</thead><tbody>{pattern_rows}</tbody></table>"
    "<p>Each extracted entity also points to its source "
    "<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>"
))

## Counting, which top-k retrieval cannot do

Three things are pinned below, and all three matter. The schema is hand-written
rather than introspected, so the model can only name labels and properties that
exist. The examples show the shape of an acceptable answer. And the prompt says
read-only, in a system where the credentials say the same thing.

Display the generated Cypher every time. It is the artifact worth reviewing.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from neo4j_graphrag.retrievers import Text2CypherRetriever

    from workshop.bedrock_providers import BedrockLLM

    hotel_schema = """
    Node properties:
    Hotel {name: STRING, address: STRING, guest_rating: FLOAT, total_rooms: INTEGER}
    Room {type: STRING, bed_configuration: STRING, max_occupancy: INTEGER}
    Amenity {name: STRING, description: STRING, fee: STRING}
    Policy {name: STRING, description: STRING}
    Service {name: STRING, description: STRING, cost: STRING, hours: STRING}
    Relationships:
    (:Hotel)-[:HAS_ROOM]->(:Room)
    (:Hotel)-[:OFFERS_AMENITY]->(:Amenity)
    (:Hotel)-[:HAS_POLICY]->(:Policy)
    (:Hotel)-[:PROVIDES_SERVICE]->(:Service)
    """
    examples = [
        "USER INPUT: What is the average rating of Paris hotels? CYPHER: MATCH (h:Hotel) WHERE toLower(h.address) CONTAINS 'paris' AND h.guest_rating IS NOT NULL RETURN avg(h.guest_rating) AS average_rating",
        "USER INPUT: How many hotels have a spa? CYPHER: MATCH (h:Hotel)-[:OFFERS_AMENITY]->(a:Amenity) WHERE toLower(a.name) CONTAINS 'spa' RETURN count(DISTINCT h) AS hotel_count",
    ]
    text2cypher_prompt = """
    Generate one read-only Cypher 25 query for the user question.
    Use only the supplied schema. Never write or delete data.
    Return only the Cypher query with no markdown fence or explanation.
    Schema:
    {schema}
    Examples:
    {examples}
    User question: {query_text}
    """
    text2cypher_retriever = Text2CypherRetriever(
        driver=driver,
        llm=BedrockLLM(region_name=os.environ.get("AWS_REGION", "us-east-1")),
        neo4j_schema=hotel_schema,
        examples=examples,
        custom_prompt=text2cypher_prompt,
    )
    structured_question = "How many hotels in the database have a swimming pool?"
    structured_result = text2cypher_retriever.search(query_text=structured_question)

    print(f"Question: {structured_question}")
    print(f"Generated Cypher:\n{structured_result.metadata['cypher']}\n")
    print("Returned records:")
    for item in structured_result.items:
        print(f"  {item.content}")
    print("\nWhy this fits: the database computes the count over all matching relationships.")

## The same pattern behind a trust boundary

`Text2CypherRetriever` ran in this notebook's process, holding your database
credentials, with a schema you pinned by hand. That is fine for a notebook and
is not what you would ship.

Moving it behind a governed service, so schema pinning and read-only enforcement
live server-side and the agent calls a tool rather than holding credentials, is
a change of trust boundary and not a change of capability. Lab 5 takes the other
route and deploys the fixed `HybridCypherRetriever` instead, because a fixed
traversal has nothing to enforce at request time.

## Which pattern should I use?

| Query shape | Start with | Why |
|---|---|---|
| Semantic lookup or paraphrase | `VectorRetriever` | Meaning matters more than exact wording |
| Exact name, policy term, or identifier | `HybridRetriever` | Combines semantic and full-text signals |
| Either of those, plus connected context | The `Cypher` variant of each | One reviewed traversal, no model in the query path |
| Aggregation or count | `Text2CypherRetriever` | Neo4j computes over the full matching set |
| A question the graph has no facts for | No retriever can invent coverage | Return an empty result and let the agent decline |

## A note on chunking

This workshop uses large chunks during extraction so each hotel stays intact
while its entities and relationships are created. Production systems often keep
that extraction strategy and create separate, smaller retrieval chunks. The
right retrieval chunk size depends on the source material and the query shapes,
and it is not tuned here.

---

**Next:** [Lab 3: Agents and tools](../03-agents-and-tools/) picks up
`search_hotel_knowledge`, the fixed hybrid retriever from `2.2`, and gives it to
an agent named `hotel_agent` alongside lifecycle hooks and a booking tool.
Nothing from this optional notebook carries forward: `Text2CypherRetriever` is
not part of the deployed path.

In [ ]:
if RETRIEVAL_READY:
    driver.close()
    print("Connection closed.")